# Here we will train the forum Classifiers and set their Thresholds  

In [ ]:
import os
import pandas as pd
from turtle import pd


import torch
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import BertTokenizer, BertForSequenceClassification, AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from datetime import datetime
from tqdm import tqdm
import time
from bs4 import BeautifulSoup,MarkupResemblesLocatorWarning
from Forums_Entity import Forum_Entity


import torch
from tqdm.autonotebook import tqdm
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import MinMaxScaler
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
#Configs
PATH_TO_DATA=f"""../results/stackexchange_{{sub}}_combined/{{llm}}/aligned.parquet"""
NUMBER_OF_WEEKS=53
METRIC='normalized_ViewCount'
MODEL_NAME = "bert-base-uncased"
BATCH_SIZE = 100
EPOCHS = 3
LR = 1e-5
MAX_LEN = 64
SUBJECTS = ['math','stackoverflow','ubuntu','english','latex']
DATA_START = pd.to_datetime("2024-04-23")
DATA_END=DATA_START + pd.DateOffset(months=3)
LLMS = ['EleutherAI-pythia-6.9b']
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Using device: {device}")
print(f"📊 Training configuration: Batch size={BATCH_SIZE}, Epochs={EPOCHS}, LR={LR}, Max length={MAX_LEN}")
forum = Forum_Entity(model_name="", num_labels=2)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LR)
NUMBER_OF_CLASSIFFIERS = 3

# Improving Our classifier
We can see that there is a decline in the results of the classifier, maybe we can see if we can improve it by giving higher threshold

In [ ]:



class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item
    
    
def train_model(model, train_loader, optimizer, criterion,epoch_num, total_epochs):
    model.train()
    total_loss = 0
    
    with tqdm(train_loader, desc=f"Epoch {epoch_num}/{total_epochs} - Training", unit="batch") as pbar:
        for batch in pbar:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            
            # Update progress bar with current loss
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return total_loss / len(train_loader)

def eval_model(model, eval_loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    
    # Lists to store all labels and predictions for final AUC calculation
    all_labels = []
    all_preds_prob = []
    
    with torch.no_grad():
        with tqdm(eval_loader, desc="Validation", unit="batch") as pbar:
            for batch in pbar:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                outputs = model(input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = criterion(logits, labels)
                total_loss += loss.item()

                # Get predicted class
                preds = torch.argmax(logits, dim=1)
                
                # Get predicted probabilities for AUC calculation
                preds_prob = torch.softmax(logits, dim=1)[:, 1]

                # Store labels and predictions for final calculation
                all_labels.append(labels.cpu())
                all_preds_prob.append(preds_prob.cpu())
                
                # Update accuracy metrics
                correct += (preds == labels).sum().item()
                total += labels.size(0)
                
    # Concatenate all stored tensors
    all_labels = torch.cat(all_labels)
    all_preds_prob = torch.cat(all_preds_prob)
    
    # Calculate final AUC on the entire dataset
    # This check ensures it won't fail if the full dataset only has one class
    if len(all_labels.unique()) > 1:
        final_auc = roc_auc_score(all_labels, all_preds_prob)
    else:
        # If the entire dataset has only one class, AUC is undefined
        final_auc = float('nan') 
    
    # Calculate final loss and accuracy
    final_loss = total_loss / len(eval_loader)
    final_acc = correct / total if total > 0 else 0
    
    print(f"Validation Loss: {final_loss:.4f} | Accuracy: {final_acc:.4f} | AUC: {final_auc:.4f}")
    
    return final_loss, final_auc,all_preds_prob

def preprocess_text(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove HTML tags from 'title' and 'body' columns of the dataframe.

    Args:
        df (pd.DataFrame): Input dataframe with 'title' and 'body' columns.

    Returns:
        pd.DataFrame: Cleaned dataframe with HTML removed.
    """
    df = df.copy()
    df.rename({'QuestionCreationDate': 'CreationDate',
                    'QuestionTitle': 'title',
                    'QuestionBody': 'body'}, axis=1, inplace=True)
    for col in ['Title', 'Body', 'title', 'body']:
        if col in df.columns:
            df[col] = df[col].astype(str).apply(lambda x: BeautifulSoup(x, "html.parser").get_text())
    if 'Title' in df.columns and 'Body' in df.columns:
        df['text'] = df['Title'].fillna('') + '\n' + df['Body'].fillna('')        
    else:
        df['text'] = df['title'].fillna('') + '\n' + df['body'].fillna('')
    return df




    

## Create dataset

In [ ]:
for llm in LLMS:
    all_genAI_data=[]
    for sub in SUBJECTS:
        print(f'Loading data for subject: {sub}')
        
        # Load genAI data
        genAI_data_sub = pd.read_parquet(PATH_TO_DATA.format(sub=sub, llm=llm))
        if 'CreationDate' not in genAI_data_sub.columns or 'ViewCount' not in genAI_data_sub.columns:
            genAI_data_sub = genAI_data_sub.rename(columns={'Question_Creation_Date': 'CreationDate', 'QuestionViewCount': 'ViewCount'})
        genAI_data_sub["CreationDate"] = pd.to_datetime(genAI_data_sub["CreationDate"])
        genAI_data_sub=genAI_data_sub[genAI_data_sub['CreationDate']>=DATA_START]
        genAI_data_sub['subject'] = sub  # Add subject identifier
        all_genAI_data.append(genAI_data_sub)
    # Concatenate all dataframes
    sub='united'
    genAI_data = pd.concat(all_genAI_data, ignore_index=True)
    genAI_data['year_week'] = genAI_data['CreationDate'].dt.to_period('W')
    genAI_data['normalized_ViewCount'] = MinMaxScaler().fit_transform(genAI_data[['ViewCount']])
    
    

In [15]:
import numpy as np
from sklearn.metrics import precision_recall_curve
def find_threshold_precision_twice_recall(y_true, y_probs, ratio=2.0):
    """
    Finds the threshold where Precision >= ratio * Recall,
    and among those, picks the threshold with the highest Recall.
    
    Parameters:
    -----------
    y_true : array-like of shape (n_samples,)
        True binary labels.
    y_probs : array-like of shape (n_samples,)
        Predicted probabilities for the positive class.
    ratio : float, default=2.0
        Required Precision / Recall ratio.
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)

    # thresholds aligns with recalls[1:], precisions[1:]
    precisions, recalls, thresholds = precisions[1:], recalls[1:], thresholds

    # mask for the constraint: P >= ratio * R
    mask = precisions >= ratio * recalls

    if not np.any(mask):
        print(f"No threshold found where Precision >= {ratio} × Recall.")
        return 0.5  # fallback default

    # among feasible points, maximize recall
    best_idx = np.argmax(recalls[mask])
    chosen = np.where(mask)[0][best_idx]

    best_threshold = thresholds[chosen]
    best_precision = precisions[chosen]
    best_recall = recalls[chosen]

    print(f"Chosen Threshold={best_threshold:.4f}, Precision={best_precision:.4f}, Recall={best_recall:.4f} (Precision ≈ {ratio}× Recall)")
    return best_threshold




In [ ]:

for i in range (NUMBER_OF_CLASSIFFIERS):
    model = AutoModelForSequenceClassification.from_pretrained(f"bert-base-uncased", num_labels=2).to(device)
    DATA_END = DATA_START + pd.DateOffset(months=3*(i+1))
    df=genAI_data.copy()[genAI_data['CreationDate']<=DATA_END]
    print(f"📅 Training on data up to: {DATA_END.date()} | Total samples: {len(df)}")
    threshold = df[METRIC].median()
    std = df[METRIC].std()

    # 1. Define the 'Inner' limits
    inner_low = df[METRIC].quantile(0.4) 
    inner_high = df[METRIC].quantile(0.6) 

    # 2. Grab values OUTSIDE this range
    # We want: (Too Small) OR (Too Big)
    df_filtered = df[
        (df[METRIC] < inner_low) | 
        (df[METRIC] > inner_high)
    ].copy()

    print(f"Filtered dataset size: {len(df_filtered)}")
    # Relabel
    df_filtered["target"] = (df_filtered[METRIC] > threshold).astype(int)
    df = df_filtered

    positive_ratio = df["target"].mean()
    print(f"🎯 Target ratio: {positive_ratio:.2%} positive")
    print(f"📊 Class distribution:\n{df['target'].value_counts()}")
    # -------------------------
    # Dataset creation
    # -------------------------
    dataset = TextDataset(
        df["text"].tolist(),
        df["target"].tolist(),
        tokenizer,
        max_len=MAX_LEN
    )

    split = int(0.8 * len(dataset))
    train_dataset, eval_dataset = random_split(dataset, [split, len(dataset) - split])

    print(
        f"🔀 Train/Val sizes for {sub}: "
        f"{len(train_dataset)}/{len(eval_dataset)} samples"
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False)
    eval_loader = DataLoader(eval_dataset, batch_size=BATCH_SIZE)

    # -------------------------
    # Initialize model
    # -------------------------
    print("🧠 Initializing BERT model...")
    model = BertForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=LR)
    criterion = torch.nn.CrossEntropyLoss()

    # -------------------------
    # Training loop
    # -------------------------
    for epoch in range(EPOCHS):
        epoch_start = time.time()

        train_loss = train_model(
            model, train_loader, optimizer, criterion, epoch + 1, EPOCHS
        )
        val_loss, val_auc, probs = eval_model(model, eval_loader, criterion)

    
    # Get predictions from the classifier
    # Access the original data using the indices stored in the subset
    forum.model=model
    eval_labels = [eval_dataset.dataset.labels[i] for i in eval_dataset.indices]
    eval_texts = [eval_dataset.dataset.texts[i] for i in eval_dataset.indices]
    y_pred_probs = pd.Series(forum.predict_proba_batch(eval_texts)).apply(lambda x: x[1])
    threshold=find_threshold_precision_twice_recall(eval_labels, y_pred_probs.values, ratio=2.0)
    print(
            f"📈 {sub} | "
            f"Threshold: {threshold:.4f} | Val Loss: {val_loss:.4f} | AUC: {val_auc:.4f}"
        )

    # -------------------------
    # Save per-subject model
    # -------------------------
    save_dir = f"Player-F-Models"
    os.makedirs(save_dir, exist_ok=True)

    model.save_pretrained(save_dir+f"/{sub}_model_{i}")
    tokenizer.save_pretrained(save_dir+f"/{sub}_model")

    print(f"💾 Saved {sub} model → {save_dir}")

    print("\n🎉 Training completed for all subjects!")
    print("=" * 80)


In [ ]:
Thresholds={
    "united_model_0": 0.8188 ,
    "united_model_1": 0.9005,
    "united_model_2": 0.8396 ,
    "united_model_3": 0.8734,
}

In [43]:
# Access the original data using the indices stored in the subset
eval_labels = [eval_dataset.dataset.labels[i] for i in eval_dataset.indices]
eval_texts = [eval_dataset.dataset.texts[i] for i in eval_dataset.indices]